In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "sentence-transformers"
])

In [ ]:
import sentence_transformers

print("sentence-transformers version:", sentence_transformers.__version__)
print("Installation successful!")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Semantic NLP model loaded successfully!")

In [ ]:
RISK_PROTOTYPES = {
    "scam": [
        "This profile is involved in scams and fraud",
        "This person is trying to deceive people",
        "This account is suspicious and fraudulent"
    ],

    "money": [
        "Make money quickly through this opportunity",
        "Invest money and earn high profits",
        "Get rich quickly with this offer"
    ],

    "urgency": [
        "Act immediately before this opportunity expires",
        "Limited time opportunity, act now",
        "This is your last chance"
    ],

    "contact": [
        "Contact me through WhatsApp or Telegram",
        "Send me a direct message for more information",
        "DM me to get this offer"
    ],

    "promotional": [
        "Win a free prize or giveaway",
        "Get a special bonus or discount",
        "Free promotional offer"
    ]
}

print("Risk categories created:")
print(list(RISK_PROTOTYPES.keys()))

In [ ]:
prototype_embeddings = {}

for category, sentences in RISK_PROTOTYPES.items():
    embeddings = model.encode(sentences)
    prototype_embeddings[category] = embeddings

print("Prototype embeddings generated successfully!")

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

In [ ]:
def calculate_semantic_scores(text):

    if not text or str(text).strip() == "":
        return {}

    text_embedding = model.encode(str(text))

    category_scores = {}

    for category, embeddings in prototype_embeddings.items():

        similarities = []

        for embedding in embeddings:

            similarity = cosine_similarity(
                text_embedding,
                embedding
            )

            similarities.append(similarity)

        category_scores[category] = max(similarities)

    return category_scores

In [ ]:
test_bios = [
    "Win free money! DM me on Telegram",
    "Computer science student",
    "Limited time investment offer, contact me now",
    "Travel photography and mountains",
    "Double your cash with this amazing opportunity",
    "I love coding and building machine learning projects"
]

for bio in test_bios:

    scores = calculate_semantic_scores(bio)

    print("\nBio:", bio)

    for category, score in scores.items():
        print(f"{category}: {score:.3f}")

In [ ]:
SEMANTIC_MIN_SIMILARITY = 0.25
SEMANTIC_MAX_SIMILARITY = 0.60
MATCH_THRESHOLD = 0.40

print("Semantic risk scoring configuration loaded.")

In [ ]:
def similarity_to_risk_score(similarity):
    
    if similarity <= SEMANTIC_MIN_SIMILARITY:
        return 0.0
    
    if similarity >= SEMANTIC_MAX_SIMILARITY:
        return 100.0
    
    score = (
        (similarity - SEMANTIC_MIN_SIMILARITY)
        / (SEMANTIC_MAX_SIMILARITY - SEMANTIC_MIN_SIMILARITY)
    ) * 100
    
    return round(score, 2)

In [ ]:
def calculate_semantic_risk(text):

    category_scores = calculate_semantic_scores(text)

    if not category_scores:
        return {
            "nlp_risk_score": 0.0,
            "matched_categories": [],
            "category_scores": {},
            "highest_risk_category": None,
            "highest_similarity": 0.0
        }

    max_category = max(
        category_scores,
        key=category_scores.get
    )

    max_similarity = category_scores[max_category]

    risk_score = similarity_to_risk_score(max_similarity)

    matched_categories = [
        category
        for category, score in category_scores.items()
        if score >= MATCH_THRESHOLD
    ]

    # No meaningful category if similarity is below threshold
    if max_similarity < MATCH_THRESHOLD:
        highest_risk_category = None
    else:
        highest_risk_category = max_category

    return {
        "nlp_risk_score": risk_score,
        "matched_categories": matched_categories,
        "category_scores": category_scores,
        "highest_risk_category": highest_risk_category,
        "highest_similarity": round(max_similarity, 3)
    }

In [19]:
test_bios = [
    "Win free money! DM me on Telegram",
    "Computer science student",
    "Limited time investment offer, contact me now",
    "Travel photography and mountains",
    "Double your cash with this amazing opportunity",
    "I love coding and building machine learning projects"
]

for bio in test_bios:
    
    result = calculate_semantic_risk(bio)
    
    print("\n" + "=" * 60)
    print("Bio:", bio)
    print("NLP Risk Score:", result["nlp_risk_score"])
    print("Matched Categories:", result["matched_categories"])
    print("Highest Risk Category:", result["highest_risk_category"])
    print("Highest Similarity:", result["highest_similarity"])


Bio: Win free money! DM me on Telegram
NLP Risk Score: 82.76
Matched Categories: ['contact', 'promotional']
Highest Risk Category: contact
Highest Similarity: 0.54

Bio: Computer science student
NLP Risk Score: 0.0
Matched Categories: []
Highest Risk Category: None
Highest Similarity: 0.215

Bio: Limited time investment offer, contact me now
NLP Risk Score: 75.23
Matched Categories: ['money', 'urgency', 'contact', 'promotional']
Highest Risk Category: money
Highest Similarity: 0.513

Bio: Travel photography and mountains
NLP Risk Score: 0.0
Matched Categories: []
Highest Risk Category: None
Highest Similarity: 0.151

Bio: Double your cash with this amazing opportunity
NLP Risk Score: 96.0
Matched Categories: ['money']
Highest Risk Category: money
Highest Similarity: 0.586

Bio: I love coding and building machine learning projects
NLP Risk Score: 0.0
Matched Categories: []
Highest Risk Category: None
Highest Similarity: 0.211
